# 模型类

到目前为止，我们已经构建了五个核心基础类：
**张量**、**层**、**损失函数**、**优化器**、**数据集**。
它们各自负责一件事，但使用时需要手动把它们串联起来——
创建对象、传参、调用 `zero_grad`、`backward`、`step`……

本章引入最后一个核心类：**模型**（Model）。
它把层、损失函数、优化器，以及完整的训练和测试流程，
全部封装在一起，让使用者只需两行代码：

```python
model.train(dataset, epochs=1000)
predictions, loss = model.test(dataset)
```

同时，本章引入**轮次**（Epoch）的概念，让模型可以对全部训练数据反复迭代，直至收敛。

---

完成这章后，整个框架的分层结构如下：

```{figure} images/model.png
:align: center
:width: 360px
**图例：框架分层结构**

* **顶层**：模型（流程控制）；
* **中层**：层、损失函数、优化器（逻辑执行）；
* **底层**：张量（自动微分）。

```

In [ ]:
from abc import ABC, abstractmethod
import numpy as np

## 基础架构

### 张量

In [ ]:
class Tensor:

    def __init__(self, data):
        self.data = np.array(data, dtype=np.float64)
        self.grad = np.zeros_like(self.data)
        self.gradient_fn = None
        self.parents = set()

    def backward(self):
        if self.gradient_fn is not None:
            self.gradient_fn()
        for parent in self.parents:
            parent.backward()

    @property
    def size(self):
        return self.data.shape[-1]

    def __repr__(self):
        return f'Tensor({self.data})'

### 基础数据集

In [ ]:
class Dataset(ABC):

    def __init__(self, batch_size=1):
        self.batch_size = batch_size
        self.train_features = None
        self.train_labels   = None
        self.test_features  = None
        self.test_labels    = None
        self.load()
        self.features = self.train_features
        self.labels   = self.train_labels

    @abstractmethod
    def load(self):
        pass

    def train(self):
        self.features = self.train_features
        self.labels   = self.train_labels

    def eval(self):
        self.features = self.test_features
        self.labels   = self.test_labels

    @property
    def shape(self):
        return Tensor(self.features).size, Tensor(self.labels).size

    def items(self):
        return Tensor(self.features), Tensor(self.labels)

    def __len__(self):
        return len(self.features) // self.batch_size

    def __getitem__(self, index):
        start = index * self.batch_size
        end   = start + self.batch_size
        return Tensor(self.features[start:end]), Tensor(self.labels[start:end])

### 基础层

In [ ]:
class Layer(ABC):

    def __call__(self, x: Tensor):
        return self.forward(x)

    @abstractmethod
    def forward(self, x: Tensor):
        pass

    @property
    def parameters(self):
        return []

    def __repr__(self):
        return f"{type(self).__name__}[]"

### 基础损失函数

In [ ]:
class Loss(ABC):

    def __call__(self, p: Tensor, y: Tensor):
        return self.loss(p, y)

    @abstractmethod
    def loss(self, p: Tensor, y: Tensor):
        pass

### 基础优化器

In [ ]:
class Optimizer(ABC):

    def __init__(self, parameters, lr):
        self.parameters = parameters
        self.lr = lr

    def zero_grad(self):
        """将所有参数的梯度清零，在每轮训练开始前调用。"""
        for param in self.parameters:
            param.grad = np.zeros_like(param.data)

    @abstractmethod
    def step(self):
        pass

### 基础模型

**基础模型**（Base Model）是一个抽象类，定义了模型的两个核心接口：

* **`train(dataset, epochs)`**：对训练集反复迭代 `epochs` 轮，每轮遍历全部批次；
* **`test(dataset)`**：在测试集上评估模型，返回预测值和损失。

创建一个模型需要提供三样东西：**层**、**损失函数**、**优化器**。

``💡 当前 Model 只接受单个层。支持多层网络需要一个能把多个层串联起来的结构——
这将在下一章引入 Sequential（顺序模型）时解决。``

In [ ]:
class Model(ABC):

    def __init__(self, layer, loss_fn, optimizer):
        self.layer     = layer
        self.loss_fn   = loss_fn
        self.optimizer = optimizer

    @abstractmethod
    def train(self, dataset, epochs):
        pass

    @abstractmethod
    def test(self, dataset):
        pass

## 数据

### 数据集（冰激凌销量）

In [ ]:
class LRDataset(Dataset):   # 注意：继承 Dataset，不要与基类同名

    def load(self):
        self.train_features = [[22.5, 72.0],
                               [31.4, 45.0],
                               [19.8, 85.0],
                               [27.6, 63.0]]
        self.train_labels   = [[95],
                               [210],
                               [70],
                               [155]]
        self.test_features  = [[28.1, 58.0]]
        self.test_labels    = [[165]]

## 模型

### 线性层

In [ ]:
class Linear(Layer):

    def __init__(self, in_size, out_size):
        self.weight = Tensor(np.ones((out_size, in_size)) / in_size)
        self.bias   = Tensor(np.zeros(out_size))

    def forward(self, x: Tensor):
        p = Tensor(x.data @ self.weight.data.T + self.bias.data)

        def gradient_fn():
            self.weight.grad += p.grad.T @ x.data
            self.bias.grad   += np.sum(p.grad, axis=0)
            x.grad           += p.grad @ self.weight.data

        p.gradient_fn = gradient_fn
        p.parents = {x}
        return p

    @property
    def parameters(self):
        return [self.weight, self.bias]

    def __repr__(self):
        return f'Linear[weight{self.weight.data.shape}; bias{self.bias.data.shape}]'

### 均方误差损失函数

In [ ]:
class MSELoss(Loss):

    def loss(self, p: Tensor, y: Tensor):
        mse = Tensor(np.mean(np.square(y.data - p.data)))

        def gradient_fn():
            n = len(y.data)
            p.grad += -2 * (y.data - p.data) / n

        mse.gradient_fn = gradient_fn
        mse.parents = {p}
        return mse

### 随机梯度下降优化器

In [ ]:
class SGDOptimizer(Optimizer):

    def step(self):
        for param in self.parameters:
            param.data -= param.grad * self.lr

### 神经网络模型

`NNModel` 是第一个具体的模型类，把完整的训练和测试流程封装在内。

**`train()` 方法**实现了标准的训练循环：
外层按 `epochs` 轮次迭代，内层按批次遍历训练集。
每批次执行固定的四步：`zero_grad → 前向传播 → backward → step`。
为了便于观察收敛过程，每隔 `epochs // 10` 轮打印一次当前批次的损失。

**`test()` 方法**切换到测试模式，用全部测试集做一次前向传播，
返回预测值和测试损失。

In [ ]:
class NNModel(Model):

    def train(self, dataset, epochs):
        dataset.train()  # 切换到训练模式
        log_interval = max(1, epochs // 10)

        for epoch in range(epochs):
            epoch_loss = 0
            for i in range(len(dataset)):
                features, labels = dataset[i]

                self.optimizer.zero_grad()                    # 1. 清零梯度
                predictions = self.layer(features)            # 2. 前向传播
                loss = self.loss_fn(predictions, labels)
                loss.backward()                               # 3. 自动微分
                self.optimizer.step()                         # 4. 更新参数
                epoch_loss += float(loss.data)

            if epoch % log_interval == 0 or epoch == epochs - 1:
                avg = epoch_loss / len(dataset)
                print(f'epoch {epoch:4d}/{epochs}  train_loss={avg:.4f}')

    def test(self, dataset):
        dataset.eval()  # 切换到测试模式
        features, labels = dataset.items()
        predictions = self.layer(features)
        loss = self.loss_fn(predictions, labels)
        return predictions, loss

## 设置

### 学习率

In [ ]:
LEARNING_RATE = 0.00001

### 批大小

In [ ]:
BATCH_SIZE = 2

### 轮次

**轮次**（Epoch）是指对训练集完整遍历一次。
设为 `1000` 轮，即每个训练样本会被模型学习 `1000` 次。
轮次越多，参数越接近最优值，但过多可能导致过拟合。

In [ ]:
EPOCHS = 1000

## 训练

### 建模

建模流程与上一章相同：分别创建数据集、层、损失函数和优化器，
再将它们作为参数传入模型。

In [ ]:
dataset   = LRDataset(BATCH_SIZE)

layer     = Linear(dataset.shape[0], dataset.shape[1])
loss_fn   = MSELoss()
optimizer = SGDOptimizer(layer.parameters, lr=LEARNING_RATE)

model = NNModel(layer, loss_fn, optimizer)
print(model.layer)

### 迭代训练

现在只需一行代码启动训练，模型会自动完成 1000 轮迭代，
并每 100 轮打印一次训练损失，让我们直观看到收敛过程。

In [ ]:
model.train(dataset, EPOCHS)

## 验证

### 测试

训练完成后，一行代码切换到测试模式并评估：

In [ ]:
predictions, loss = model.test(dataset)
print(f'prediction: {predictions}')
print(f'test loss:  {loss}')

经过 1000 轮训练，模型预测值约为 `163.5`，真实值为 `165`，测试损失约 `2.18`。

对比第六章只训练 1 轮时的测试损失 `11839`，提升超过 5000 倍——
这清楚地展示了多轮迭代训练的价值。

---

至此，我们的神经网络训练框架已经可以完整运作：
从原始数据出发，经过多轮批次训练，最终输出接近真实值的预测结果。

但当前模型只有一层线性变换，表达能力有限。
要构建真正的深层神经网络，我们需要能把多个层串联起来的机制。
下一章将引入**顺序模型**（Sequential）和**激活函数**，把网络加深。

## 课后练习

1. 在 `NNModel.train()` 里，每轮结束后调用 `model.test(dataset)`，
   同时打印训练损失和测试损失。观察两者的差距随训练轮次如何变化。

2. 尝试增加训练轮次（如 5000 轮），观察测试损失是否继续下降，
   还是在某个点后开始上升（过拟合）。

3. 扩展 `NNModel`，增加一个**早停**（Early Stopping）机制：
   如果测试损失连续 N 轮不再下降，则提前停止训练。